In [ ]:
import pandas as pd

df = pd.read_csv('final_merged_data.csv',  keep_default_na=True, delimiter=',', skipinitialspace=True)
df_sample = pd.read_csv(file_path, nrows=5)  
df_sample.head()



,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,address,lat,...,min_humidity_quality_indicator,min_relative_humidity_percent,humidity_std_quality_indicator,relative_humidity_std_deviation,max_pressure_quality_indicator,max_barometric_pressure_hpa,min_pressure_quality_indicator,min_barometric_pressure_hpa,pressure_std_quality_indicator,barometric_pressure_std_deviation
0,2024-12-01 00:10:00,10,15,1,True,True,True,DAME STREET,Dame Street,53.344006,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
1,2024-12-01 00:10:00,100,17,8,True,True,True,HEUSTON BRIDGE (SOUTH),Heuston Bridge (South),53.347107,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
2,2024-12-01 00:10:00,109,20,9,True,True,True,BUCKINGHAM STREET LOWER,Buckingham Street Lower,53.353333,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
3,2024-12-01 00:10:00,11,1,29,True,True,True,EARLSFORT TERRACE,Earlsfort Terrace,53.334293,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
4,2024-12-01 00:10:00,114,4,36,True,True,True,WILTON TERRACE (PARK),Wilton Terrace (Park),53.333652,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083


In [5]:
print(df_sample.columns.tolist())

['last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'address', 'lat', 'lon', 'capacity', 'stno', 'year', 'month', 'day', 'hour', 'minute', 'max_air_temp_quality_indicator', 'max_air_temperature_celsius', 'min_air_temp_quality_indicator', 'min_air_temperature_celsius', 'air_temp_std_quality_indicator', 'air_temperature_std_deviation', 'max_grass_temp_quality_indicator', 'max_grass_temperature_celsius', 'min_grass_temp_quality_indicator', 'min_grass_temperature_celsius', 'grass_temp_std_quality_indicator', 'grass_temperature_std_deviation', 'max_soil_temp_5cm_quality_indicator', 'max_soil_temperature_5cm_celsius', 'min_soil_temp_5cm_quality_indicator', 'min_soil_temperature_5cm_celsius', 'soil_temp_std_5cm_quality_indicator', 'soil_temperature_std_deviation_5cm', 'max_soil_temp_10cm_quality_indicator', 'max_soil_temperature_10cm_celsius', 'min_soil_temp_10cm_quality_indicator', 'min_soil_temperature_10cm_cels

In [ ]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# === 设置路径 ===
file_path = "final_merged_data.csv"
model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

# === 加载数据（只需最简字段） ===
use_cols = ['last_reported', 'station_id',
            'num_bikes_available', 'num_docks_available']
df = pd.read_csv(file_path, usecols=use_cols)
df['last_reported'] = pd.to_datetime(df['last_reported'])

# === 时间特征 ===
df['hour'] = df['last_reported'].dt.hour
df['weekday'] = df['last_reported'].dt.weekday
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# === 获取站点列表 ===
station_ids = df['station_id'].unique()
print(f"共检测到 {len(station_ids)} 个站点")

# === 对每个站点训练模型 ===
for station_id in station_ids:
    df_station = df[df['station_id'] == station_id].dropna()
    if len(df_station) < 100:
        print(f"⏭️ 跳过站点 {station_id}（数据不足）")
        continue

    features = ['hour', 'weekday', 'is_weekend']
    targets = ['num_bikes_available', 'num_docks_available']

    X = df_station[features]
    y = df_station[targets]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae_bikes = mean_absolute_error(y_test['num_bikes_available'], y_pred[:, 0])
    mae_docks = mean_absolute_error(y_test['num_docks_available'], y_pred[:, 1])

    model_path = os.path.join(model_dir, f"model_station_{station_id}.pkl")
    joblib.dump(model, model_path)

    print(f"✅ 模型完成 - 站点 {station_id}（MAE: bikes={mae_bikes:.2f}, docks={mae_docks:.2f}）")


共检测到 115 个站点
✅ 模型完成 - 站点 10（MAE: bikes=1.79, docks=1.77）
✅ 模型完成 - 站点 100（MAE: bikes=3.41, docks=3.36）
✅ 模型完成 - 站点 109（MAE: bikes=5.64, docks=5.66）
✅ 模型完成 - 站点 11（MAE: bikes=4.82, docks=4.82）
✅ 模型完成 - 站点 114（MAE: bikes=5.42, docks=5.47）
✅ 模型完成 - 站点 116（MAE: bikes=5.66, docks=5.67）
✅ 模型完成 - 站点 13（MAE: bikes=6.34, docks=5.91）
✅ 模型完成 - 站点 14（MAE: bikes=5.19, docks=5.19）
✅ 模型完成 - 站点 15（MAE: bikes=1.29, docks=1.28）
✅ 模型完成 - 站点 17（MAE: bikes=4.54, docks=4.54）
✅ 模型完成 - 站点 18（MAE: bikes=5.17, docks=5.15）
✅ 模型完成 - 站点 19（MAE: bikes=4.73, docks=4.73）
✅ 模型完成 - 站点 2（MAE: bikes=1.73, docks=1.73）
✅ 模型完成 - 站点 20（MAE: bikes=3.99, docks=4.00）
✅ 模型完成 - 站点 22（MAE: bikes=3.89, docks=3.89）
✅ 模型完成 - 站点 24（MAE: bikes=4.18, docks=4.17）
✅ 模型完成 - 站点 28（MAE: bikes=5.45, docks=5.45）
✅ 模型完成 - 站点 29（MAE: bikes=5.44, docks=5.44）
✅ 模型完成 - 站点 3（MAE: bikes=4.10, docks=4.10）
✅ 模型完成 - 站点 30（MAE: bikes=2.04, docks=2.04）
✅ 模型完成 - 站点 31（MAE: bikes=5.01, docks=5.02）
✅ 模型完成 - 站点 33（MAE: bikes=4.99, docks=4.99）
✅ 模型完成 - 站点 34（MA